# **Length Estimator Timing**

Measures the time the trained length classifier (`03_length_estimator`) takes to estimate the output length of new requests, on a T4 GPU and on a CPU. The result is saved in `estimator_timing.json` and read by the experiments notebook.

## Configuration and data

In [1]:
import json, os, time
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

EST_DIR = "data/request_estimates"
REQUEST_POOL_PATH = "data/request_pool/request_pool_v2.parquet"
N_TIMED = 1000          # requests timed
BATCH_SIZE = 32         # as in the estimator notebook
MAX_TOKENS_READ = 512   # as in the estimator notebook
SEED = 42

meta = json.load(open(os.path.join(EST_DIR, "request_estimates_v1_metadata.json")))
bucket_value = np.array(meta["bucket_values"])
pool = pd.read_parquet(REQUEST_POOL_PATH)
pool = pool[~pool["source"].isin(meta["excluded_sources"])]
sample = pool.sample(min(N_TIMED, len(pool)), random_state=SEED)
texts = sample["text"].tolist()
tok = AutoTokenizer.from_pretrained(os.path.join(EST_DIR, "length_classifier"))
print(f"{len(texts)} requests to time | GPU available: {torch.cuda.is_available()}")

Mounted at /content/drive
1000 requests to time | GPU available: True


## Timing

In [2]:
half = (MAX_TOKENS_READ - 2) // 2

def encode(batch):
    ids = tok(list(batch), add_special_tokens=False, truncation=False)["input_ids"]
    ids = [i if len(i) <= 2 * half else i[:half] + i[-half:] for i in ids]
    ids = [[tok.cls_token_id] + i + [tok.sep_token_id] for i in ids]
    width = max(len(i) for i in ids)
    return (torch.tensor([i + [tok.pad_token_id] * (width - len(i)) for i in ids]),
            torch.tensor([[1] * len(i) + [0] * (width - len(i)) for i in ids]))

@torch.no_grad()
def estimate(model, device):
    out = []
    for s in range(0, len(texts), BATCH_SIZE):
        input_ids, attention = encode(texts[s:s + BATCH_SIZE])     # reading the text is part of the time
        logits = model(input_ids=input_ids.to(device), attention_mask=attention.to(device)).logits
        out.append(logits.argmax(-1).cpu().numpy())
    return bucket_value[np.concatenate(out)]

result = {"n_requests": len(texts), "batch_size": BATCH_SIZE}
for device in (["cuda", "cpu"] if torch.cuda.is_available() else ["cpu"]):
    model = AutoModelForSequenceClassification.from_pretrained(os.path.join(EST_DIR, "length_classifier")).to(device).eval()
    estimate(model, device)                                    # warm-up: first call loads kernels, not timed
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    pred = estimate(model, device)
    if device == "cuda":
        torch.cuda.synchronize()
    seconds = time.perf_counter() - t0
    key = "gpu" if device == "cuda" else "cpu"
    result[f"{key}_seconds"] = seconds
    result[f"{key}_ms_per_request"] = 1000 * seconds / len(texts)
    print(f"{key.upper()}: {seconds:.2f} s for {len(texts)} requests = {result[f'{key}_ms_per_request']:.2f} ms per request")

if "DistilBERT" in meta["chosen"]:     # the saved estimates come from this classifier: they must be the same
    check = pd.read_parquet(os.path.join(EST_DIR, "request_estimates_v1.parquet")).set_index("request_id")
    same = np.mean(np.round(pred).astype(int) == check.loc[sample["request_id"], "predicted_output_tokens"].values)
    print(f"Same estimates as the saved file: {same:.1%} of the timed requests (should be 100%)")
    result["same_as_saved_estimates"] = float(same)
else:
    print(f"The saved estimates come from '{meta['chosen']}', not from the classifier: no comparison")
with open(os.path.join(EST_DIR, "estimator_timing.json"), "w") as f:
    json.dump(result, f, indent=2)
print("Saved", os.path.join(EST_DIR, "estimator_timing.json"))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (608 > 512). Running this sequence through the model will result in indexing errors


GPU: 14.71 s for 1000 requests = 14.71 ms per request


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

CPU: 504.60 s for 1000 requests = 504.60 ms per request
Same estimates as the saved file: 100.0% of the timed requests (should be 100%)
Saved /content/drive/MyDrive/thesis/request_estimates/estimator_timing.json
